# **Model Implementation**

Training and Testing Data

In [ ]:
x = df.drop('y', axis=1)
y = df['y']

In [ ]:
df.to_csv("Data/final_data.csv", index=False)

Splitting the dataset into training and testing

In [ ]:
# We implement stratified split because of class imbalance.

from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
x_train = pd.get_dummies(x_train, drop_first=True)
x_test = pd.get_dummies(x_test, drop_first=True)

# Aligning columns to ensure that the training and test datasets
# have exactly the same feature columns in the same order before making predictions.

# This happens as we have done one hot encoding
x_train, x_test = x_train.align(x_test, join='left', axis=1, fill_value=0)

There is class imbalance, so we implement class weighting, i.e. Instead of changing the dataset, we adjust the models.

In [ ]:
y_train.value_counts(normalize=True)

In [ ]:
# Scaling the data for logisitic Regresssion
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()

x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

### **Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(x_train, y_train)


In [ ]:
y_pred = lr.predict(x_test)
y_prob = lr.predict_proba(x_test)[:,1]

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

In [ ]:
cm = confusion_matrix(y_test, y_pred)                # Confusion Matrix
print("Confusion Matrix:\n", cm)


print("\nClassification Report:\n")                   # Precision, Recall, F1
print(classification_report(y_test, y_pred))


roc_auc = roc_auc_score(y_test, y_prob)               #ROC-AUC score
print("ROC-AUC Score:", roc_auc)


In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure()
plt.plot(fpr, tpr, label="Logistic Regression (AUC = %0.2f)" % roc_auc)
plt.plot([0,1],[0,1],'--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")

plt.legend()
plt.show()

# TPR - True Poritive Rate or Recall
# FPR - False Positive Rate


Inference

The confusion matrix shows how the model distinguishes between risky and non-risky customers.
A low False Negative rate is important, since missing risky customers could lead to financial losses for the bank.



Precision - High precision means that when the model flags a customer as risky, it is usually correct. This reduces unnecessary investigation of low-risk customers.



Rrecall ( PTR ) - Out of all actual risky customers, how many the model detected.



F1 Score - Balanced measure of precision and recall.